#### Ahora que tengo el csv limpio, empiezo con los modelos de machine learning

In [51]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import pickle

nba = pd.read_csv('nba_agrupado.csv')

In [41]:
# divido el dataframe en variables predictoras (X) y variable objetivo (y)
X = nba[['age', 'gp', 'pts_media_carrera', 'reb', 'ast', 'net_rating', 'ts_pct']]
y = nba['pts_22_23']

In [42]:
# divido los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=11)

In [43]:
# imprimo el tamaño de los conjuntos de entrenamiento y prueba
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(405, 7) (46, 7) (405,) (46,)


#### Ya tengo dividido el dataframe en las variables y en la parte de entrenamiento y de test. Empiezo con un modelo sencillo de regresión lineal. Lo entreno, hago predicciones y evalúo.

In [44]:
# llamo al modelo de regresión lineal
modelo_basico_regresion = LinearRegression()
# entreno el modelo con los datos de entrenamiento
modelo_basico_regresion.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [45]:
# hago predicciones con el modelo entrenado
prediccion_modelo_basico_regresion = modelo_basico_regresion.predict(X_test)

In [46]:
# evalúo el modelo con las métricas de regresión

print(f'MAE: {mean_absolute_error(y_test, prediccion_modelo_basico_regresion)}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, prediccion_modelo_basico_regresion))}')
print(f'R2: {r2_score(y_test, prediccion_modelo_basico_regresion)}')

MAE: 2.991541474560017
RMSE: 3.6238837080289708
R2: 0.6503624142070736


#### Primeras conclusiones:

* MAE: 2.99. Aquí se podría decir que el modelo se equivoca de media 3 puntos por partido. Si un jugador anota 20, predice entre 17 y 23 aproximadamente.
* RMSE: 3.62. Esta métrica penaliza errores más grandes. Si fuese mucho mayor que el MAE, sig parecido al MAE pero penaliza más los errores grandes. Si es parecido al MAE significa que las predicciones son uniformes.
* R²: 0.65. El modelo explica el 65% de la variación de los puntos. No es mal punto de partida.

In [47]:
# llamo al modelo de regresión Ridge para repartir pesos entre las variables 
modelo_ridge = Ridge(alpha=1.0) # alpha es el parámetro de regularización, puedes ajustarlo para mejorar el rendimiento del modelo
# entreno el modelo
modelo_ridge.fit(X_train, y_train)
# hago predicciones con el modelo entrenado
prediccion_ridge = modelo_ridge.predict(X_test)
# evalúo el modelo
print(f'MAE: {mean_absolute_error(y_test, prediccion_ridge)}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, prediccion_ridge))}')
print(f'R2: {r2_score(y_test, prediccion_ridge)}')

MAE: 2.9892224331730985
RMSE: 3.6191275425334495
R2: 0.6512795755160581


#### Ahora que he evaluado Ridge, parece que el primer modelo no tenía ninguna variable que dominase en exceso sobre las demás. Ahora bien, hemos asumido que la relación entre variables y target es lineal y no tiene por qué ser así. Para comprobar si se puede mejorar las predicciones, podríamos probar con Random Forest. Aquí se detectarán patrones más complejos.

In [48]:
# llamo al modelo de regresión de Random Forest
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=11) # utilizo 100 árboles
# entreno el modelo
modelo_rf.fit(X_train, y_train)
# hago predicciones
prediccion_rf = modelo_rf.predict(X_test)
# evalúo el modelo
print(f'MAE: {mean_absolute_error(y_test, prediccion_rf)}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, prediccion_rf))}')
print(f'R2: {r2_score(y_test, prediccion_rf)}')

MAE: 2.730195652173913
RMSE: 3.4096898207774036
R2: 0.6904724188528526


#### Esto mejora. Ahora el modelo solo se equivoca 2,7 puntos de media. Y el modelo ya explica el 69% de la variación de los puntos. Probaré, finalmente, con XGBoost.

In [49]:
# llamo al modelo de regresión XGBoost
modelo_xgb = XGBRegressor(n_estimators=100, random_state=11)
# entreno el modelo
modelo_xgb.fit(X_train, y_train)
# hago predicciones
prediccion_xgb = modelo_xgb.predict(X_test)
# evalúo el modelo
print(f'MAE: {mean_absolute_error(y_test, prediccion_xgb)}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, prediccion_xgb))}')
print(f'R2: {r2_score(y_test, prediccion_xgb)}')

MAE: 3.2530618232229482
RMSE: 4.055197164572671
R2: 0.5621820563295477


#### Este modelo empeora un poco los resultados del anterior. Hago una comparativa final para poner los datos sobre la mesa y hacer predicciones con el mejor modelo en el siguiente notebook.

In [50]:
resultados = pd.DataFrame({
    'Modelo': ['Regresión lineal', 'Ridge', 'Random Forest', 'XGBoost'],
    'MAE': [mean_absolute_error(y_test, prediccion_modelo_basico_regresion),
            mean_absolute_error(y_test, prediccion_ridge),
            mean_absolute_error(y_test, prediccion_rf),
            mean_absolute_error(y_test, prediccion_xgb)],
    'RMSE': [np.sqrt(mean_squared_error(y_test, prediccion_modelo_basico_regresion)),
             np.sqrt(mean_squared_error(y_test, prediccion_ridge)),
             np.sqrt(mean_squared_error(y_test, prediccion_rf)),
             np.sqrt(mean_squared_error(y_test, prediccion_xgb))],
    'R2': [r2_score(y_test, prediccion_modelo_basico_regresion),
           r2_score(y_test, prediccion_ridge),
           r2_score(y_test, prediccion_rf),
           r2_score(y_test, prediccion_xgb)]
})

resultados

,Modelo,MAE,RMSE,R2
0,Regresión lineal,2.991541,3.623884,0.650362
1,Ridge,2.989222,3.619128,0.651280
2,Random Forest,2.730196,3.409690,0.690472
3,XGBoost,3.253062,4.055197,0.562182


#### Pruebo a hiperparametrizar el Random Forest para ver si puedo dar con un mejor modelo.

In [52]:
# creo los parámetros para el GridSearchCV
parametros = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10]
}
# hago el GridSearchCV para encontrar los mejores hiperparámetros del modelo de Random Forest
grid_search = GridSearchCV(RandomForestRegressor(random_state=11), 
                           parametros, 
                           cv=5, 
                           scoring='neg_mean_absolute_error',
                           n_jobs=-1)
# entreno el GridSearchCV
grid_search.fit(X_train, y_train)
# imprimo los mejores parámetros encontrados
print('Mejores parámetros:', grid_search.best_params_)

Mejores parámetros: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}


#### Ahora entreno un Random Forest con esos parámetros

In [53]:
# entreno el modelo de Random Forest con los mejores hiperparámetros encontrados
modelo_rf_optimizado = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=10, random_state=11)
# entreno el modelo
modelo_rf_optimizado.fit(X_train, y_train)
# hago predicciones
prediccion_rf_optimizado = modelo_rf_optimizado.predict(X_test)

print(f'MAE: {mean_absolute_error(y_test, prediccion_rf_optimizado)}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, prediccion_rf_optimizado))}')
print(f'R2: {r2_score(y_test, prediccion_rf_optimizado)}')

MAE: 2.6734830046825038
RMSE: 3.367280704895109
R2: 0.6981242331799324


#### Tampoco hemos mejorado tanto. Guardo el modelo y paso a las predicciones y a la comparación con los resultados reales.

In [55]:
# guardo el modelo de Random Forest
pickle.dump(modelo_rf_optimizado, open('modelo_rf_optimizado.pkl', 'wb'))